In [22]:
import xarray as xr
import glob
from dask_jobqueue import PBSCluster
from dask.distributed import Client
import numpy as np

In [34]:
def crop_ds(ds, border=[-90, 90, -180, 180]):
    ds.coords['lon'] = (ds.coords['lon'] + 180) % 360 - 180
    ds = ds.sortby(ds.lon)
    ds = ds.sortby(ds.lat)
    ds_crop = ds.sel(lat=slice(border[0], border[1]), lon=slice(border[2], border[3]))
    return ds_crop


def ONI(sstanom):
    sstanom = crop_ds(sstanom, [-5, 5, -170, -120])
    index = sstanom.mean(('lon', 'lat')).resample(time='ME').mean('time').rolling(time=1).mean('time')
    return index


def ATL3(sstanom):
    sstanom = crop_ds(sstanom, [-3, 3, -20, 0])
    index = sstanom.mean(('lon', 'lat')).resample(time='ME').mean('time').rolling(time=1).mean('time')
    return index

In [24]:
cluster = PBSCluster(
    cores=1,
    memory='16GB',
    processes=1,
    queue='casper',
    local_directory='$TMPDIR',
    account='P93300313',
    walltime='1:00:00'
)
cluster.scale(jobs=10)
client = Client(cluster)
client

/glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 41227 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/41227/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/41227/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.182:33747,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/41227/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [25]:
files = sorted(glob.glob('/gdex/data/d277007/avhrr_v2.1/*/*'))
sstds = xr.open_mfdataset(files, combine='nested', concat_dim='time',
                          engine='h5netcdf', parallel=True,
                          chunks={})
sstds

<xarray.Dataset> Size: 271GB
Dimensions:  (time: 16331, zlev: 1, lat: 720, lon: 1440)
Coordinates:
  * time     (time) datetime64[ns] 131kB 1981-09-01T12:00:00 ... 2026-05-18T1...
  * zlev     (zlev) float32 4B 0.0
  * lat      (lat) float32 3kB -89.88 -89.62 -89.38 -89.12 ... 89.38 89.62 89.88
  * lon      (lon) float32 6kB 0.125 0.375 0.625 0.875 ... 359.4 359.6 359.9
Data variables:
    anom     (time, zlev, lat, lon) float32 68GB dask.array<chunksize=(1, 1, 720, 1440), meta=np.ndarray>
    err      (time, zlev, lat, lon) float32 68GB dask.array<chunksize=(1, 1, 720, 1440), meta=np.ndarray>
    ice      (time, zlev, lat, lon) float32 68GB dask.array<chunksize=(1, 1, 720, 1440), meta=np.ndarray>
    sst      (time, zlev, lat, lon) float32 68GB dask.array<chunksize=(1, 1, 720, 1440), meta=np.ndarray>
Attributes: (12/37)
    title:                      NOAA/NCEI 1/4 Degree Daily Optimum Interpolat...
    source:                     ICOADS, NCEP_GTS, GSFC_ICE, NCEP_ICE, Pathfin...
    id:                         oisst-avhrr-v02r01.19810901.nc
    naming_authority:           gov.noaa.ncei
    summary:                    NOAAs 1/4-degree Daily Optimum Interpolation ...
    cdm_data_type:              Grid
    ...                         ...
    metadata_link:              https://doi.org/10.25921/RE9P-PT57
    ncei_template_version:      NCEI_NetCDF_Grid_Template_v2.0
    comment:                    Data was converted from NetCDF-3 to NetCDF-4 ...
    sensor:                     Thermometer, AVHRR
    Conventions:                CF-1.6, ACDD-1.3
    references:                 Reynolds, et al.(2007) Daily High-Resolution-...

In [26]:
sstads = sstds['anom']
sstads

<xarray.DataArray 'anom' (time: 16331, zlev: 1, lat: 720, lon: 1440)> Size: 68GB
dask.array<concatenate, shape=(16331, 1, 720, 1440), dtype=float32, chunksize=(1, 1, 720, 1440), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 131kB 1981-09-01T12:00:00 ... 2026-05-18T1...
  * zlev     (zlev) float32 4B 0.0
  * lat      (lat) float32 3kB -89.88 -89.62 -89.38 -89.12 ... 89.38 89.62 89.88
  * lon      (lon) float32 6kB 0.125 0.375 0.625 0.875 ... 359.4 359.6 359.9
Attributes:
    long_name:  Daily sea surface temperature anomalies
    valid_min:  -1200
    valid_max:  1200
    units:      Celsius

In [35]:
oni_ds = ONI(sstads)
atl3_ds = ATL3(sstads)

In [49]:
oni_ds = oni_ds.rename('ONI_ERA5').chunk({'time': -1}).isel(zlev=0, drop=True)
oni_ds

<xarray.DataArray 'ONI_ERA5' (time: 537)> Size: 2kB
dask.array<getitem, shape=(537,), dtype=float32, chunksize=(537,), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 4kB 1981-09-30 1981-10-31 ... 2026-05-31
Attributes:
    long_name:  Daily sea surface temperature anomalies
    valid_min:  -1200
    valid_max:  1200
    units:      Celsius

In [50]:
atl3_ds = atl3_ds.rename('ATL3_ERA5').chunk({'time': -1}).isel(zlev=0, drop=True)
atl3_ds

<xarray.DataArray 'ATL3_ERA5' (time: 537)> Size: 2kB
dask.array<getitem, shape=(537,), dtype=float32, chunksize=(537,), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 4kB 1981-09-30 1981-10-31 ... 2026-05-31
Attributes:
    long_name:  Daily sea surface temperature anomalies
    valid_min:  -1200
    valid_max:  1200
    units:      Celsius

In [51]:
oni_ds.to_netcdf('/glade/work/acruz/Caribbean_Heat_data/OISST/ONI.nc')
atl3_ds.to_netcdf('/glade/work/acruz/Caribbean_Heat_data/OISST/ATL3.nc')

/glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/distributed/client.py:3398: UserWarning: Sending large graph of size 34.23 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
/glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/distributed/client.py:3398: UserWarning: Sending large graph of size 34.23 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


In [52]:
client.shutdown()